# Estimating Fatigue from Facial Video during Incremental Cycling

**Part B — Group project .** Camille Jaffeux, Ancelin Gely — Master 2 IEAP.

**Objectives** :
1) Test whether simple facial features follow perceived exertion, heart rate and the intensity domains.
2) Test the link between facial features and ventilatory thresholds.

**Design.**
Before the protocol, a stress test with gas exchange (masked test, v1) has been done to identify the ventilatory
thresholds VT1 and VT2. This masked test is used **only** for the thresholds and is not analysed further.

Two participants, each performing a maximal incremental cycling test with the same
protocol (rest ≈ 2 min, warm-up 3 min, 1-min stages to exhaustion, recovery): **v2** where the whole face is
visible. ECG and perceived exertion (Borg CR10, at the end of each stage) are recorded in every test.

The notebook detects the face
(Viola–Jones), stabilises it, extracts three features (mouth opening, forehead redness, head instability),
aggregates them per second and tests whether they change with effort and track Borg CR10, heart rate and the
intensity domains given by VT1 and VT2.

| § | Content | § | Content |
|---|---|---|---|
| 0 | Fatigue, facial signs, hypotheses, outcomes | 6 | Per-second features → `features.csv` |
| 1 | Setup and parameters | 7 | Descriptive statistics and time course |
| 2 | Data | 8 | Rest vs effort vs recovery |
| 3 | Face detection, ROIs, per-frame features | 9 | Dose–response, **primary outcome**, normalised curves, intensity domains |
| 4 | ECG and synchronisation | 10–11 | Discussion, reproducibility |
| 5 | Quality control | | |

## 0. Fatigue, facial signs, hypotheses and outcomes

### 0.1 What we mean by "fatigue"
"Fatigue" is not a single variable. Enoka and Duchateau (2016) describe it as a symptom in which physical and
cognitive function is limited by the interaction between **performance fatigability** (decline in an objective
measure of performance) and **perceived fatigability** (change in the sensations that regulate the integrity of
the performer). Four meanings are usually distinguished:

* **Muscular (neuromuscular) fatigue** — exercise-induced reduction in the ability of a muscle to produce force or
  power (Gandevia 2001). It needs a force or power test and has no specific facial signature: not measured here
  (task failure at the end of each test is our only, crude, performance marker).
* **Cardiovascular strain** — the rise in heart rate needed to meet the metabolic demand; strictly internal load,
  but the main objective marker of proximity to maximal capacity. Measured with the ECG, and located relative to
  **VT1, VT2 and V̇O₂max** measured by gas exchange in the masked test (v1).
* **Mental fatigue** — caused by prolonged demanding cognitive activity (Marcora et al. 2009); not induced by a
  10-minute ramp. Slow blinks and eyelid closure belong to the drowsiness literature, not to exercise.
* **Perceived exertion** — "the feeling of how heavy and strenuous a physical task is" (Borg 1998), rated on the
  Borg CR10 scale at the end of each stage.

**Operational definition.** "Fatigue" = the **acute exertion** accumulated during a maximal incremental test,
measured as **cardiovascular strain** (ECG heart rate, intensity domain) and **perceived exertion** (Borg CR10).
Facial signs should reflect it through ventilation (mouth opening) and skin blood flow (redness). Head motion
is a mechanical control.

### 0.2 Facial signs

| Sign | Status | Evidence | CV feature (course) |
|---|---|---|---|
| Mouth opening | **key — primary outcome** | *Mouth open* had the highest repeated-measures correlation with RPE (r = 0.70) and, with *jaw drop*, best predicted RPE in 113 students on an incremental cycling test (Timme & Brand 2020); mouth open–close rate rose with intensity (Khanal et al. 2018) | dark-cavity segmentation in a mouth ROI, opening/closing, area / face area (C1–C2) |
| Redness | **key** | Forehead colour related to heart rate during a ramp in 9 students, with large between-subject heterogeneity (Khanal et al. 2022) | mean a\* (Lab) on the lower forehead, skin pixels only (as Khanal) (C1) |
| Head instability | **key — mechanical control** | Facial movement rises non-linearly with power (Miles et al. 2018) | jitter of the raw Viola–Jones box centre (C3) |


### 0.3 Hypotheses 
* **H1 : ventilatory** : mouth-opening area increases with stage, Borg CR10 and heart rate and stays elevated in early recovery.
* **H2 : skin blood flow** : forehead redness increases with heart rate.
* **H3 : mechanical control** : head instability rises with power and returns to baseline as soon as pedalling stops.
* **H4 : ventilatory thresholds** : Mouth opening increases more steeply above VT1 and VT2 transition (from nose to mouth breathing)

### 0.4 Outcomes (pre-specified)
**Primary outcome** : within-test Spearman correlation between **mouth-opening area** and **Borg CR10** and between **mouth-opening area** and heart rate

**Secondary outcomes** : within-test Spearman correlation between **forehead redness** and **heart rate**, between
**forehead redness** and **power**, and between **head instability** and **power**.

## 1. Setup and parameters

In [ ]:
# Import Libraries
import os, sys, json, time, datetime, hashlib, platform, warnings
from collections import deque

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import signal, stats
from scipy.ndimage import maximum_filter1d

The following code sets up the notebook and the face detection and tracking parameters. Viola-jones face detection is used to detect the face in each frame. It requires several parameters :

**Viola-Jones parameters :**
* `det_width` : width of the downscaled frame used for detection.
* `scale_factor` : how much the image size is reduced at each image scale.
* `min_neighbors` : minimum number of neighbors required to consider a detection valid
* `min_face_frac` : minimum fraction of the frame that must be covered by a detected face
* `full_search_every` : how often to perform a full search for faces (in frames)
* `smooth_n` : number of frames used for smoothing the face position
* `max_hold_frames` : maximum number of frames to hold a face before considering it lost

**Regions of interest (ROIs) :**
Regions of interest are defined as fractions of the stabilised face box. The following ROIs are used :
* forehead : redness (a\*)
* cheeks (right and left) : reference brightness for the mouth threshold
* mouth : mouth opening

These Region are defined by fraction as (x0, y0, x1, y1) of the stabilised face box. It permits to locate the ROIs in the face box and extract features from them.

**Colour parameters for skin detection and mouth opening :**
* `skin_cr` : range of Cr values for skin detection in YCrCb colour space
* `skin_cb` : range of Cb values for skin detection in YCrCb colour space
* `min_skin_frac` : minimum fraction of the ROI that must be classified as skin
* `ref_init_s` : duration (s) at the start of the video used as gray-card reference (only if a gray card is filmed)
* `mouth_dark_k` : factor by which the mouth is considered darker than the cheeks

YCrCB is a colour space that separates luminance (Y) from chrominance (Cr and Cb). It differs from HSV in that it separates luminance from chrominance, while HSV separates hue, saturation and value. YCrCb is often used in video compression and broadcasting, while HSV is often used in computer vision and image processing.

**Quality control :**
* `flow_size` : size of the face crop used for the sharpness and for the head motion that checks the ECG–video synchronisation. 
* `sharp_rel_min` : a frame is counted as blurred when its sharpness is below this fraction of the median at rest

**ECG and heart rate parameters :** 
Permits to go from the ECG signal to heart rate and to detect the R peaks. The parameters are : 
* ecg_band_hz : bandpass filter for the ECG signal - keeps frequencys between 5 and 15 Hz
* rr_range_s : range of RR intervals (s) to consider valid - keeps RR intervals between 0.27 and 2 s
* rr_max_dev : maximum deviation of RR intervals from the median to consider valid - keeps RR intervals that deviate less than 20% from the median
* hr_max_gap_s : maximum gap between heart rate measurements to consider valid - keeps gaps less than 3 s 

**analysis windows** 
Windows are used to compare the features between rest, effort and recovery. It takes the mean of the features in a window of time. windows are defined as the last 120 s of rest, the last 60 s of effort and the first 120 s of recovery. 

In [ ]:
# Display parameters
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160, "display.max_columns", 40)

# Notebook parameters
DEMO = False                # True: synthetic test data; False: data from filmed sessions
ANONYMIZE_FIGURES = True    # pixelate faces in figures
SEED = 42                   # Random seed for reproducibility

# Directories and parameters
ROOT = os.path.abspath(".")
DATA_DIR = os.path.join(ROOT, "data_demo" if DEMO else "data")
OUT_DIR = os.path.join(ROOT, "outputs_demo" if DEMO else "outputs")
FIG_DIR, TAB_DIR, CACHE_DIR = (os.path.join(OUT_DIR, d) for d in ("figures", "tables", "cache"))
for d in (FIG_DIR, TAB_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)

# Face detection and tracking parameters, skin colour thresholds, analysis windows, and statistics
PARAMS = dict(
    # Viola-Jones and stabilisation
    det_width=480, scale_factor=1.1, min_neighbors=5, min_face_frac=0.20, full_search_every=30,
    smooth_n=9, max_hold_frames=12,

    # regions of interest: fractions (x0, y0, x1, y1) of the stabilised face box
    roi=dict(forehead=(0.30, 0.08, 0.70, 0.22), cheek_l=(0.14, 0.52, 0.34, 0.70), cheek_r=(0.66, 0.52, 0.86, 0.70),
             mouth=(0.27, 0.66, 0.73, 0.97)),

    # colour parameters for skin detection and mouth opening
    skin_cr=(133, 180), skin_cb=(77, 135), min_skin_frac=0.20, ref_init_s=2.0,
    mouth_dark_k=0.30,                        # mouth cavity = darker than k x cheek brightness (lips excluded)

    # quality control: face crop size (sharpness, head motion for the ECG synchronisation check), blur threshold
    flow_size=128, sharp_rel_min=0.5,

    # ecg and heart rate parameters
    ecg_band_hz=(5.0, 15.0), rr_range_s=(0.27, 2.0), rr_max_dev=0.20, hr_max_gap_s=3.0,

    # analysis windows (s) and statistics
    rest_window_s=120, effort_window_s=60, recovery_window_s=120, stage_window_s=30, min_stage_s=15,
    block_s=10, min_valid_s=7, n_boot=2000, n_perm=9999,
)

# Features and columns
FEATURE_INFO = {   # name: (label, unit, family)
    "mouth_area": ("Mouth opening area", "% face area", "key"),
    "forehead_a": ("Forehead redness a*", "a* units", "key"),
    "head_jitter": ("Head instability (box jitter)", "% face width", "key"),
    "hr_ecg": ("ECG heart rate", "bpm", "reference"),
}
FEATURES = [f for f in FEATURE_INFO if f != "hr_ecg"]
ALL = FEATURES + ["hr_ecg"]
KEY = [f for f in FEATURES if FEATURE_INFO[f][2] == "key"]
# per-frame columns: raw box centre (head jitter), stabilised box, features, and technical columns
# (skin_frac, cheek_val = reference of the mouth threshold, sharpness, flow_rigid = synchronisation check)
FRAME_COLS = ["raw_cx", "raw_cy", "box_x", "box_y", "box_w", "box_h", "forehead_a", "skin_frac", "cheek_val",
              "mouth_area", "sharpness", "flow_rigid"]
PARAM_HASH = hashlib.md5(json.dumps(PARAMS, sort_keys=True).encode()).hexdigest()[:8]
json.dump(dict(PARAMS=PARAMS, SEED=SEED, DEMO=DEMO), open(os.path.join(OUT_DIR, "params.json"), "w"), indent=2)

# Plot style: fixed colour per condition / per test, recessive grid, one axis per panel.
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
COND = {"rest": "#2a78d6", "effort": "#eb6834", "recovery": "#1baf7a"}
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "axes.edgecolor": INK2,
    "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
    "axes.titlesize": 9, "axes.titleweight": "bold", "lines.linewidth": 1.5, "legend.frameon": False, "figure.dpi": 110})


def show(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches="tight")
    plt.show()


def label(f):
    return ("★ " if FEATURE_INFO[f][2] == "key" else "") + FEATURE_INFO[f][0]


print("OpenCV", cv2.__version__, "| params", PARAM_HASH, "| data:", DATA_DIR)
assert hasattr(cv2, "CascadeClassifier"), ("this OpenCV has no Viola-Jones detector (removed in OpenCV 5): run  "
                                          '%pip install "opencv-python>=4.8,<5"  in a cell, then restart the kernel')
print("mouth threshold:", PARAMS["mouth_dark_k"], "x cheek brightness (mouth area must be about 0 at rest)")

## 2. Data
**`data/tests.xlsx`** (template: `data/tests_template.xlsx`) describes the recordings; the videos are never cut.
* Sheet `tests` — one row per test: 
  * `session_id`, 
  * `subject_id`, 
  * `video_file`, `ecg_file`, 
  * `video times typed as `mm:ss` — `t_rest_start, t_warmup_start, t_first_stage`,
  * `t_effort_end, t_recovery_end` — with `first_stage` (number of the first stage whose start is in the video) and
  * `stage_s` (stage duration); `ecg_offset_s` (**t_video = t_ecg + offset**, §4); `vt1_W, vt2_W` (power at the
  thresholds, once per subject); 

* Sheet `paliers` — one row per segment and test: `session_id, palier (REPOS, WARMUP, P1 … Pn, RECUP), power_W,
  borg_cr10`.


In [ ]:
# Convert video times typed as mm:ss, h:mm:ss or seconds to seconds (None if empty).
def seconds(v):
    """Video time typed as mm:ss, h:mm:ss or seconds -> seconds (None if empty)."""
    if isinstance(v, datetime.time):                               # Excel stored 02:05 as a clock time: read it as mm:ss
        return float(v.hour * 60 + v.minute if v.second == 0 else v.hour * 3600 + v.minute * 60 + v.second)
    if v is None or pd.isna(v) or str(v).strip() == "":
        return None
    return float(sum(float(x) * 60 ** k for k, x in enumerate(reversed(str(v).replace(",", ".").split(":")))))

# Load the tests and segments from the two sheets of tests.xlsx, converting video times to seconds.
def load_tests_xlsx(path):
    """Sessions (filmed tests) and segments in video time from the two sheets of tests.xlsx."""
    T, P = (pd.read_excel(path, sheet_name=s).dropna(how="all") for s in ("tests", "paliers"))
    T, P = (d[d.session_id.notna() & ~d.session_id.astype(str).str.upper().str.startswith("EXEMPLE")].copy() for d in (T, P))
    for d in (T, P):
        d["session_id"] = d.session_id.astype(str).str.strip()
    T["subject_id"] = T.subject_id.astype(str).str.strip()
    T[["vt1_W", "vt2_W"]] = T.groupby("subject_id")[["vt1_W", "vt2_W"]].transform("first")    # typed once per subject
    P = P.dropna(subset=["power_W", "borg_cr10"], how="all").copy()
    code = P.palier.astype(str).str.strip().str.upper()
    P["condition"] = np.select([code.isin(["REPOS", "REST"]), code.str.contains("WARM|CHAUF"), code.str.startswith("REC")],
                               ["rest", "warmup", "recovery"], "effort")
    P["stage"] = pd.to_numeric(code.str.lstrip("P"), errors="coerce")                         # "P7" or "7" -> 7
    eff = P.condition == "effort"
    pmax = eff & P.stage.isna()                                                                # "PMAX" = last stage
    P.loc[pmax, "stage"] = (P[eff].groupby("session_id").stage.transform("max") + 1).reindex(P.index)[pmax]
    rows = []
    for r in T[T.video_file.notna()].itertuples():
        pal, L = P[P.session_id == r.session_id], 60.0 if pd.isna(r.stage_s) else float(r.stage_s)
        t0, t_end = seconds(r.t_first_stage), seconds(r.t_effort_end)
        assert t0 is not None and t_end is not None, f"{r.session_id}: t_first_stage and t_effort_end are required"
        t1 = t0 - ((1 if pd.isna(r.first_stage) else int(r.first_stage)) - 1) * L     # start of stage 1 (< 0: before the video)
        tr, tw = seconds(r.t_rest_start), seconds(r.t_warmup_start)
        stages = sorted(pal[pal.condition == "effort"].stage.dropna().astype(int))
        segs = [("rest", 0, tr, t1 if tw is None else tw), ("warmup", np.nan, tw, t1)]
        segs += [("effort", k, t1 + (k - 1) * L, min(t1 + k * L, t_end)) for k in stages]
        segs += [("recovery", -1, t_end, seconds(r.t_recovery_end))]
        for cond, k, a, b in segs:
            if a is None or b is None or b <= max(a, 0):           # not in the video
                continue
            q = pal[(pal.condition == cond) & ((pal.stage == k) | (cond != "effort"))]
            power = q.power_W.iloc[0] if len(q) and pd.notna(q.power_W.iloc[0]) else (np.nan if cond == "effort" else 0)
            rows.append(dict(session_id=r.session_id, segment=f"P{k}" if cond == "effort" else cond, condition=cond, stage=k,
                             t_start_s=max(a, 0.0), t_end_s=b, power_W=power,
                             borg_cr10=q.borg_cr10.iloc[0] if len(q) else np.nan))
        over = t_end - (t1 + (stages[-1] if stages else 0) * L)
        if not stages or over > 5 or over <= -L:
            print(f"CHECK {r.session_id}: effort ends {over:+.0f} s after the end of the last stage listed in 'paliers' "
                  f"({len(stages)} stages) -> check t_first_stage, first_stage, t_effort_end and the stage rows")
    S = T[T.video_file.notna()].rename(columns={"video_file": "video_path", "ecg_file": "ecg_path"})
    return S, pd.DataFrame(rows)


def opt(row, name, default=None):
    """Optional cell of a sessions.csv row (missing column, NaN or empty string -> default)."""
    v = row.get(name, default)
    return default if v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip() == "" else v


def parse_box(s):
    return None if s is None else tuple(float(v) for v in str(s).split(","))


if DEMO and not os.path.exists(os.path.join(DATA_DIR, "sessions.csv")):
    sys.path.insert(0, os.path.join(ROOT, "tools"))
    import make_synthetic_session
    make_synthetic_session.make_all(DATA_DIR, seed=1)          # two synthetic tests (15-20 min)

if os.path.exists(os.path.join(DATA_DIR, "tests.xlsx")):
    sessions, segments = load_tests_xlsx(os.path.join(DATA_DIR, "tests.xlsx"))
else:
    sessions = pd.read_csv(os.path.join(DATA_DIR, "sessions.csv"), dtype={"session_id": str, "subject_id": str})
    segments = pd.read_csv(os.path.join(DATA_DIR, "segments.csv"), dtype={"session_id": str})


def locate(path):
    """File typed in the workbook, looked up (ignoring case) where typed, then in data/ and in data/raw/; None if absent."""
    if path is None or pd.isna(path) or not str(path).strip():
        return None
    folder, name = os.path.split(str(path).strip())
    for d in (folder, "", "raw"):
        hit = [f for f in (os.listdir(os.path.join(DATA_DIR, d)) if os.path.isdir(os.path.join(DATA_DIR, d)) else [])
               if f.lower() == name.lower()]
        if hit:
            return os.path.join(d, hit[0])
    return None


sessions = sessions.set_index("session_id", drop=False)

# The masked (gas-exchange) test only provides VT1 / VT2: the thresholds are copied to every test of the same
# subject, then the masked tests are removed from the video analysis.
for c in ("vt1_W", "vt2_W"):
    sessions[c] = pd.to_numeric(sessions[c], errors="coerce") if c in sessions else np.nan
sessions[["vt1_W", "vt2_W"]] = sessions.groupby("subject_id")[["vt1_W", "vt2_W"]].transform("first")
masked = pd.to_numeric(sessions.get("mask", pd.Series(0, index=sessions.index)), errors="coerce").fillna(0) == 1
print("masked tests (used for VT1 / VT2 only, not analysed):", list(sessions.index[masked]))
sessions = sessions[~masked].copy()
segments = segments[segments.session_id.isin(sessions.index)].reset_index(drop=True)
typed = sessions[["video_path", "ecg_path"]].copy()
sessions["video_path"], sessions["ecg_path"] = sessions.video_path.map(locate), sessions.ecg_path.map(locate)
assert sessions.video_path.notna().all(), (f"video not found: {list(typed.video_path[sessions.video_path.isna()])}; "
                                           f"files in data/: {sorted(os.listdir(DATA_DIR))}")
assert set(segments.condition) <= {"rest", "warmup", "effort", "recovery"}, "condition: rest / warmup / effort / recovery"
for sid, r in sessions.iterrows():                             # the brief requires >= 720p and >= 25 fps
    cap = cv2.VideoCapture(os.path.join(DATA_DIR, r.video_path))
    assert cap.isOpened(), f"cannot open {r.video_path} (record in H.264 'Most Compatible', HDR off)"
    w, h, fps = (cap.get(k) for k in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
    print(f"{sid}: {w:.0f}x{h:.0f} at {fps:.2f} fps, {cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps:.0f} s",
          "" if min(w, h) >= 720 and fps >= 24.9 else " <-- below the brief's requirements")
    cap.release()
display(sessions.drop(columns="session_id"))
display(segments.head(12))

## 3. Face detection, regions of interest and per-frame features
**Viola–Jones.** 
Haar cascades are used to detect the face in each frame. The cascade is trained to detect faces of size 24×24 px. The pyramid permits to detect faces of different sizes with each level being 10% smaller than the previous one. For each level, a window of fixed aspect ratio is scanned over the image and the classifier is applied to each window. 5 successive detections are needed to keep a candidate. The search is restricted to the neighbourhood of
the previous face (full frame every 30 frames or after a miss). 

Raw boxes jitter (scale quantisation): ROIs use the **median of the last 9 detections** because it gives a more stable estimate of the face position. 

Phone videos are sometimes stored sideways: the cascade only finds upright faces, so each video is tried at 0, 90, 180 and 270° on 8 frames and the orientation with the most detections is applied to every frame (or the `rotate` value typed in `tests.xlsx`). This was proposed by Claude and we decided to keep it in case we reuse the notebook for other videos.


| Column | Region | Computation | Unit |
|---|---|---|---|
| `forehead_a` | lower forehead, skin pixels (YCrCb box + opening) | mean a\* (Lab) | a\* |
| `cheek_val` | cheeks, skin pixels | mean V (HSV) — reference brightness of the mouth threshold | % |
| `mouth_area` | mouth ROI | pixels darker than 0.30 × cheek V, opening 3×3 + closing 5×5, largest central component | % face area |
| `raw_cx`, `raw_cy` | raw Viola–Jones box | centre of the box before stabilisation (head jitter, §6) | px |
| `sharpness` | face, resized 128×128 | variance of the Laplacian (quality control) | a.u. |
| `flow_rigid` | face, resized 128×128 | Farnebäck flow, median vector = head translation (only to check the ECG synchronisation, §4) | % face width/s |

The primary outcome `mouth_area` is computed in this part. Its computation follows these steps:
1. The mouth ROI is extracted from the stabilised face box.
2. The mouth ROI is converted to the HSV colour space and the V (brightness) channel is extracted.
3. The mean V of the cheeks, computed over skin pixels only (YCrCb skin mask), is multiplied by `mouth_dark_k` (0.30) to get a threshold that follows the lighting.
4. The mouth ROI is thresholded to get a binary mask of the pixels darker than this threshold (the lips stay above it).
5. Morphological opening (3×3) and closing (5×5) are applied to the mask to remove noise and fill holes.
6. Among the connected components whose centre lies in the central 20–80 % of the ROI width, the largest one is kept as the mouth cavity.
7. The area of the mouth cavity is expressed as a percentage of the face box area (0 when the mouth is closed).

In [ ]:
# Load the Haar cascade for face detection (Viola-Jones) from OpenCV or from ./cascades/.
def load_cascade(name):
    for folder in (getattr(getattr(cv2, "data", None), "haarcascades", ""), os.path.join(ROOT, "cascades")):
        c = cv2.CascadeClassifier(os.path.join(folder, name))
        if not c.empty():
            return c
    raise FileNotFoundError(f"{name}: install opencv-python 4.x or copy the XML into ./cascades/")

# Viola-Jones face detection and stabilisation parameters 
FACE_CASCADE = load_cascade("haarcascade_frontalface_default.xml")  # pre-trained Haar cascade for frontal face detection
K3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))           # kernel for morphological operations (3x3 ellipse)
K5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))           # kernel for morphological operations (5x5 ellipse)

# Security in case the video is stored sideways: rotation angles and corresponding OpenCV flags
ROT = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}
def turn(frame, rot):
    """Rotate a frame clockwise by 0, 90, 180 or 270 degrees (phone videos stored sideways)."""
    return cv2.rotate(frame, ROT[rot]) if rot else frame

# Intersection over Union (IoU) for two consecutive boxes
# bounding boxes a and b, where each box is represented as (x, y, w, h).
# Useful for evaluating the similarity of detected faces.
def iou(a, b):
    ix = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    return ix * iy / (a[2] * a[3] + b[2] * b[3] - ix * iy + 1e-9)


class FaceTracker: 
    """Viola-Jones on a downscaled gray frame (local search around the previous face) + median-of-N stabilisation."""

    # Initialize the FaceTracker with the frame dimensions and parameters.
    def __init__(self, frame_w, frame_h, p):
        self.p, self.s = p, p["det_width"] / frame_w
        self.size = (p["det_width"], int(round(frame_h * self.s)))              # resized frame for detection 
        self.buf, self.missing, self.k = deque(maxlen=p["smooth_n"]), 0, 0

    # Detect faces in the downscaled gray frame, optionally using the previous face position for a local search.
    def _detect(self, small, prev):
        p, x0, y0 = self.p, 0, 0
        kw = dict(minSize=(int(p["min_face_frac"] * min(self.size)),) * 2)       # fraction of the shorter side
        if prev is not None:                                       # search only around the previous face
            x, y, w, h = prev
            x0, y0 = int(max(0, x - 0.5 * w)), int(max(0, y - 0.5 * h))
            small = small[y0:int(y + 1.5 * h), x0:int(x + 1.5 * w)]
            kw = dict(minSize=(int(0.6 * w),) * 2, maxSize=(int(1.6 * w),) * 2)
        faces = FACE_CASCADE.detectMultiScale(small, scaleFactor=p["scale_factor"], minNeighbors=p["min_neighbors"], **kw)
        return [(x + x0, y + y0, w, h) for x, y, w, h in faces]

    # Update the tracker with a new gray frame, returning the raw and stabilised face boxes.
    def update(self, gray): # gray frame in full resolution
        """Return (raw box or None, stabilised box or None), both (x, y, w, h) in full-resolution pixels."""
        small = cv2.resize(gray, self.size, interpolation=cv2.INTER_AREA) # Interarea avoid aliasing
        prev = np.median(np.array(self.buf), axis=0) * self.s if self.buf else None
        local = prev is not None and self.k % self.p["full_search_every"] != 0
        faces = (self._detect(small, prev) if local else []) or self._detect(small, None)
        self.k += 1
        if faces:
            best = max(faces, key=lambda f: (iou(f, prev) if prev is not None else 0, f[2]))
            raw = tuple(v / self.s for v in best)
            self.buf.append(raw)
            self.missing = 0
        else:
            raw, self.missing = None, self.missing + 1
            if self.missing > self.p["max_hold_frames"]:
                self.buf.clear()
        return raw, (tuple(np.median(np.array(self.buf), axis=0)) if self.buf else None)

# Security in case the video is stored sideways: find the rotation angle that makes the face upright.
def find_rotation(path, p, n=8):
    """Clockwise rotation that makes the face upright: the one with most Viola-Jones detections in n frames."""
    cap, hits = cv2.VideoCapture(path), dict.fromkeys((0, 90, 180, 270), 0)
    for k in range(n):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int((k + 0.5) * cap.get(cv2.CAP_PROP_FRAME_COUNT) / n))
        ok, frame = cap.read()
        for rot in hits if ok else ():
            gray = cv2.cvtColor(turn(frame, rot), cv2.COLOR_BGR2GRAY)
            hits[rot] += FaceTracker(gray.shape[1], gray.shape[0], p).update(gray)[0] is not None
    cap.release()
    return max(hits, key=hits.get), hits

# create a region od interest using the coordinates of the box and the fractions provided, 
def roi(img, box, frac=(0, 0, 1, 1)):
    """Crop a ROI given as fractions (x0, y0, x1, y1) of the box (x, y, w, h), clipped to the image."""
    x, y, w, h = box
    x0, y0, x1, y1 = (max(0, int(round(v))) for v in (x + frac[0] * w, y + frac[1] * h, x + frac[2] * w, y + frac[3] * h))
    return img[y0:y1, x0:x1]

# Convert a box given as fractions of the frame dimensions to pixel coordinates.
def frac_box(frame, box):
    """Box given as fractions of the frame -> pixels."""
    H, W = frame.shape[:2]
    return box[0] * W, box[1] * H, box[2] * W, box[3] * H

# Compute the mean a* value in Lab color space and mean V value in HSV color space over the skin pixels of a BGR crop.
# The color is convert from BGR to YcrCB 
# Y = Luminance, Cr = Red-difference chroma component, Cb = Blue-difference chroma component
# YcrCB is more robust to lighting variations than RGB or HSV, making it suitable for skin detection.
def skin_colour(bgr, p):
    """Mean a* (Lab) and mean V (HSV) over the skin pixels of a BGR crop."""
    if bgr.size == 0:
        return {}
    m = cv2.inRange(cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb), (0, p["skin_cr"][0], p["skin_cb"][0]),  # detect skin pixels in YCrCb color space using thresholds for Cr and Cb channels
                    (255, p["skin_cr"][1], p["skin_cb"][1]))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, K3) > 0
    if m.mean() < p["min_skin_frac"]:
        return dict(skin_frac=m.mean())
    return dict(skin_frac=m.mean(), a=cv2.cvtColor(bgr, cv2.COLOR_BGR2Lab)[m][:, 1].mean() - 128.0, # return the mean a* value in Lab color space and mean V value in HSV color space over the skin pixels of a BGR crop
                val=cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)[m][:, 2].mean() / 2.55)

# Compute the dark mouth cavity area as a percentage of the face area, using a threshold relative to cheek brightness, morphological operations, and the largest central blob.
def mouth_area(bgr, cheek_val, face_area, p):
    """Dark mouth cavity (% of face area): threshold relative to cheek brightness, morphology, largest central blob."""
    if bgr.size == 0 or not np.isfinite(cheek_val):
        return np.nan
    m = (cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)[..., 2] < p["mouth_dark_k"] * cheek_val * 2.55).astype(np.uint8)
    m = cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_OPEN, K3), cv2.MORPH_CLOSE, K5)
    n, _, st, cen = cv2.connectedComponentsWithStats(m, 8)
    areas = [st[i, cv2.CC_STAT_AREA] for i in range(1, n) if 0.2 < cen[i][0] / m.shape[1] < 0.8]
    return 100.0 * max(areas, default=0) / face_area

# Function to extract features from a single video frame, given the stabilized face box and other parameters. 
# Computes forehead redness, skin fraction, cheek brightness, mouth area, sharpness, head motion.
def frame_features(frame, gray, prev_gray, box, gain, dt, p):
    """Features of one frame given the stabilised face box."""
    R = {k: roi(frame, box, v) for k, v in p["roi"].items()}
    if gain is not None:                                           # gray-card correction of the colour ROIs
        R = {k: np.clip(v.astype(np.float32) * gain, 0, 255).astype(np.uint8) for k, v in R.items()}
    cl, cr = R["cheek_l"], R["cheek_r"]
    cheeks = cv2.hconcat([cl, cv2.resize(cr, cl.shape[1::-1])]) if cl.size and cr.size else cl
    ch, fo = skin_colour(cheeks, p), skin_colour(R["forehead"], p)
    f = dict(forehead_a=fo.get("a", np.nan), skin_frac=fo.get("skin_frac", np.nan), cheek_val=ch.get("val", np.nan))
    f["mouth_area"] = mouth_area(R["mouth"], f["cheek_val"], box[2] * box[3], p)
    face = [roi(im, box) for im in (gray, prev_gray) if im is not None]
    if all(im.size for im in face):
        face = [cv2.resize(im, (p["flow_size"],) * 2, interpolation=cv2.INTER_AREA) for im in face]
        f["sharpness"] = float(cv2.Laplacian(face[0], cv2.CV_32F).var())
        if len(face) == 2 and dt > 0:                              # head translation, only for the ECG sync check
            flow = cv2.calcOpticalFlowFarneback(face[1], face[0], None, 0.5, 3, 15, 3, 5, 1.2, 0)
            rigid = np.median(flow.reshape(-1, 2), axis=0)
            f["flow_rigid"] = float(np.hypot(*rigid) * 100.0 / p["flow_size"] / dt)   # % face width per second
    return f

# Function to process a video file, performing face detection, stabilization, and feature extraction for each frame. The results are cached as a CSV file if specified.
def process_video(path, p, ref_roi=None, face_box=None, cache=None, rot=0):
    """One pass over the video: detection, stabilisation and per-frame features (cached as CSV)."""
    if cache and os.path.exists(cache):
        return pd.read_csv(cache)
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    tracker, rows, prev_gray, card0, t0 = None, [], None, [], time.time()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
        if rows and not t > rows[-1]["t_s"]:
            t = len(rows) / fps                                   # container without usable timestamps
        frame = turn(frame, rot)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        tracker = tracker or FaceTracker(gray.shape[1], gray.shape[0], p)
        raw, box = (None, frac_box(frame, face_box)) if face_box else tracker.update(gray)
        row = dict(frame=len(rows), t_s=t, det=raw is not None or bool(face_box))
        if raw is not None:
            row.update(raw_cx=raw[0] + raw[2] / 2, raw_cy=raw[1] + raw[3] / 2)
        gain = None
        if ref_roi:                                               # gain = card over the first seconds / card now
            card = roi(frame, frac_box(frame, ref_roi)).reshape(-1, 3).mean(0)
            if t <= p["ref_init_s"]:
                card0.append(card)
            gain = (np.mean(card0, axis=0) / np.maximum(card, 1)).astype(np.float32)
        if box is not None:
            row.update(box_x=box[0], box_y=box[1], box_w=box[2], box_h=box[3])
            row.update(frame_features(frame, gray, prev_gray, box, gain, t - rows[-1]["t_s"] if rows else 0, p))
        rows.append(row)
        prev_gray = gray
    cap.release()
    df = pd.DataFrame(rows).reindex(columns=["frame", "t_s", "det"] + FRAME_COLS)
    print(f"  {len(df)} frames in {time.time() - t0:.0f} s, detection rate {df.det.mean():.1%}")
    if cache:
        df.to_csv(cache, index=False)
    return df


FRAMES, ROTATE = {}, {}
for sid, r in sessions.iterrows():
    vpath = os.path.join(DATA_DIR, r.video_path)
    rot, hits = (int(opt(r, "rotate")), None) if opt(r, "rotate") is not None else find_rotation(vpath, PARAMS)
    ROTATE[sid] = rot
    print(f"session {sid}: frames rotated by {rot}°", "(typed)" if hits is None else f"(face found in {hits[rot]} of 8 sample frames)")
    if hits is not None and not hits[rot] and opt(r, "face_box") is None:
        print("  WARNING: no face found in any orientation -> give `rotate` and `face_box` for this test in tests.xlsx")
    sig = f"{PARAM_HASH}_{os.path.getsize(vpath)}_{int(os.path.getmtime(vpath))}_{opt(r, 'ref_roi')}_{opt(r, 'face_box')}"
    cache = os.path.join(CACHE_DIR, f"{sid}_frames_{hashlib.md5((sig + f'_rot{rot}' * bool(rot)).encode()).hexdigest()[:10]}.csv.gz")
    FRAMES[sid] = process_video(vpath, PARAMS, parse_box(opt(r, "ref_roi")), parse_box(opt(r, "face_box")), cache, rot)

## 4. ECG: R-peaks, heart rate and synchronisation
The raw ECG CSV is read with automatic detection of the separator, the time column and its unit (s, ms, µs, ns;
otherwise `ecg_fs`). When the file holds several leads (12-lead export), every lead is processed and the one with
the fewest rejected RR intervals is kept. R-peaks: Pan–Tompkins-type chain (band-pass 5–15 Hz → derivative →
squaring → 150-ms moving integration → adaptive threshold), refined on the filtered signal. RR intervals outside
0.27–2.0 s or deviating more than 20 % from the local median are rejected.

**Synchronisation** (t_video = t_ecg + `ecg_offset_s`). If `ecg_offset_s` is left empty, the heart-rate peak is
aligned with the end of effort read in the video: heart rate starts to fall within a few seconds of stopping, so
the alignment is good to about ± 5 s (negligible for 30-s stage means, to keep in mind for the recovery kinetics).
For a tighter synchronisation, tap an electrode three times in front of the camera: the tap times in ECG time are
printed below and `ecg_offset_s = t_tap_video − t_tap_ecg`. As an independent check, the motion noise of the ECG
(> 40 Hz) and the head motion of the video are both driven by pedalling and must start and stop together: the lag
that maximises their correlation is printed and the two curves are overlaid in §5.

In [ ]:
def load_ecg_csv(path, fs=None):
    """Return t (s from 0), the lead(s) as a DataFrame and the sampling rate; separator and time unit are detected."""
    df = None
    for kw in (dict(sep=";", decimal=","), dict(sep=","), dict(sep=None, engine="python")):
        try:
            d = pd.read_csv(path, **kw).apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
        except Exception:
            continue
        if d.shape[1] and d.notna().mean().max() > 0.9:
            df = d.dropna()
            break
    assert df is not None, f"cannot parse {path}: export a plain CSV with a header row"
    tcol = next((c for c in df if any(k in str(c).lower() for k in ("time", "t_s", "sec", "[ms]", "[ns]"))), None)
    X = df.drop(columns=[tcol] if tcol is not None else [])
    short = [str(c).split()[-1] for c in X]                        # ",80/mV 16 0 0 0 0  II" -> "II"
    X.columns = short if len(set(short)) == len(short) else X.columns
    if tcol is None:
        assert fs, "no time column in the ECG file: give ecg_fs (Hz) in tests.xlsx"
        return np.arange(len(X)) / float(fs), X, float(fs)
    t = df[tcol].values.astype(float)
    dt = np.median(np.diff(t))
    unit = next((u for u in (1.0, 1e-3, 1e-6, 1e-9) if 50 <= 1 / (dt * u) <= 5000), None)
    assert unit, f"time column '{tcol}' gives an implausible sampling rate"
    return (t - t[0]) * unit, X, 1 / (dt * unit)


def r_peaks(x, fs, p):
    """Indices of the R-peaks and the band-passed ECG (R waves positive)."""
    y = signal.filtfilt(*signal.butter(3, np.array(p["ecg_band_hz"]) / (fs / 2), "band"), x)
    if abs(np.percentile(y, 0.1)) > abs(np.percentile(y, 99.9)):
        y = -y
    w = max(1, int(0.15 * fs))
    integ = np.convolve(np.gradient(y) ** 2, np.ones(w) / w, mode="same")
    thr = np.maximum(0.3 * maximum_filter1d(integ, int(4 * fs)), 0.02 * np.percentile(integ, 99))
    pk, _ = signal.find_peaks(integ, height=thr, distance=int(0.27 * fs))
    half, keep = int(0.10 * fs), []
    for q in np.unique([max(0, q - half) + np.argmax(y[max(0, q - half):q + half]) for q in pk]):
        if not keep or (q - keep[-1]) / fs >= 0.27:               # refractory period on the refined peaks
            keep.append(q)
        elif y[q] > y[keep[-1]]:
            keep[-1] = q
    return np.array(keep, int), y


def heart_rate(t_r, t_max, offset, p):
    """RR table (with rejection flag) and mean heart rate for each second of VIDEO time (NaN across long gaps)."""
    rr = pd.DataFrame(dict(t=t_r[1:] + offset, rr=np.diff(t_r)))
    med = rr.rr.rolling(11, center=True, min_periods=3).median()
    rr["ok"] = rr.rr.between(*p["rr_range_s"]) & ((rr.rr - med).abs() / med <= p["rr_max_dev"])
    g = rr[rr.ok]
    grid = np.arange(0, int(t_max) + 1, 0.25)
    hr = np.interp(grid, g.t, 60 / g.rr, left=np.nan, right=np.nan)
    i = np.clip(np.searchsorted(g.t.values, grid), 1, len(g) - 1)
    hr[(g.t.values[i] - g.t.values[i - 1]) > p["hr_max_gap_s"]] = np.nan
    return rr, pd.Series(hr, np.floor(grid).astype(int)).groupby(level=0).mean().rename("hr_ecg")


def best_lead(t, X, fs, p):
    """R-peaks on every lead; keep the lead with the fewest rejected RR intervals -> (rejected share, lead, peaks, y)."""
    best = None
    for lead in X:
        idx, y = r_peaks(X[lead].values.astype(float), fs, p)
        bad = (~heart_rate(t[idx], t[-1], 0.0, p)[0].ok).mean() if len(idx) > t[-1] / 2 else 1.0     # < 30 bpm: no ECG
        if best is None or bad < best[0]:
            best = (bad, lead, idx, y)
    return best


ECG = {}
for sid, r in sessions.iterrows():
    if opt(r, "ecg_path") is None or not os.path.exists(os.path.join(DATA_DIR, r.ecg_path)):
        ECG[sid] = None
        print(f"{sid}: no ECG file -> heart rate missing for this test")
        continue
    t, X, fs = load_ecg_csv(os.path.join(DATA_DIR, r.ecg_path), opt(r, "ecg_fs"))
    bad, lead, idx, y = best_lead(t, X, fs, PARAMS)
    peak = heart_rate(t[idx], t[-1], 0.0, PARAMS)[1].rolling(5, center=True, min_periods=3).median().idxmax()
    auto = segments[(segments.session_id == sid) & (segments.condition == "effort")].t_end_s.max() - peak
    off = float(opt(r, "ecg_offset_s", auto))                      # empty: heart-rate peak = end of effort
    rr, hr = heart_rate(t[idx], FRAMES[sid].t_s.max(), off, PARAMS)
    big = np.where((t < 90) & (np.abs(y) > 2.5 * np.median(np.abs(y[idx]))))[0]      # electrode taps = big artefacts
    taps = [round(float(t[g[np.argmax(np.abs(y[g]))]]), 2) for g in np.split(big, np.where(np.diff(big) > 0.2 * fs)[0] + 1)
            if len(g)]
    hp = signal.filtfilt(*signal.butter(3, min(40, 0.35 * fs) / (fs / 2), "high"), X[lead].values.astype(float))
    noise = np.log(pd.Series(hp ** 2).groupby(np.floor(t + off).astype(int)).mean()) / 2     # log EMG / motion noise per second
    motion = FRAMES[sid].groupby(np.floor(FRAMES[sid].t_s).astype(int)).flow_rigid.mean()    # head motion per second
    r_lag = {k: np.nan_to_num(motion.corr(noise.shift(-k)), nan=-1) for k in range(-15, 16)}
    lag = max(r_lag, key=r_lag.get)                                # ECG noise is `lag` s late on the video
    ECG[sid] = dict(t=t + off, y=y, r=idx, rr=rr, hr=hr, noise=noise, motion=motion)
    print(f"{sid}: fs = {fs:.0f} Hz, lead {lead} of {X.shape[1]}, {len(idx)} R-peaks, {100 * bad:.1f} % RR rejected, "
          f"taps at t_ecg = {taps[:4]} s, HR peak at t_ecg = {peak} s\n    offset "
          f"{'typed' if opt(r, 'ecg_offset_s') is not None else 'AUTOMATIC (HR peak = end of effort)'} = {off:+.1f} s; "
          f"pedalling noise of the ECG vs head motion of the video: r = {r_lag[lag]:.2f} at lag {lag:+d} s"
          f" -> ecg_offset_s = {off - lag:+.1f} s would align them (see the figure in §5)")

## 5. Quality control
Check before trusting any feature: **(i)** the ECG (R-peaks at maximal effort, rejected beats); **(ii)** the ROIs
land on the right anatomy in every condition; **(iii)** detection rate and sharpness per condition — if detection
fails more often during effort, the frames kept are the most stable ones (data missing not at random, biased
towards no effect).

In [ ]:
def label_seconds(sec, seg, p):
    """Segment labels and analysis windows for an array of integer seconds of video time."""
    out = pd.DataFrame(dict(segment=None, condition=None, stage=np.nan, power_W=np.nan, borg_cr10=np.nan,
                            window=None, stage_window=False), index=sec)
    for s in seg.itertuples():
        m = (sec >= s.t_start_s) & (sec < s.t_end_s)
        out.loc[m, ["segment", "condition"]] = s.segment, s.condition
        out.loc[m, ["stage", "power_W", "borg_cr10"]] = s.stage, s.power_W, s.borg_cr10
        if s.condition == "effort" and s.t_end_s - s.t_start_s >= p["min_stage_s"]:   # last 30 s, within the stage
            out.loc[m & (sec >= s.t_end_s - p["stage_window_s"]), "stage_window"] = True
    t_rest, t_end = seg[seg.condition == "rest"].t_end_s.max(), seg[seg.condition == "effort"].t_end_s.max()
    out.loc[(sec >= t_rest - p["rest_window_s"]) & (sec < t_rest) & (out.condition == "rest").values, "window"] = "rest"
    out.loc[(sec >= t_end - p["effort_window_s"]) & (sec < t_end), "window"] = "effort"
    out.loc[(sec >= t_end) & (sec < t_end + p["recovery_window_s"]), "window"] = "recovery"
    out.loc[(out.window == "rest").values, "stage_window"] = True  # rest = stage 0 of the dose-response
    out["t_since_end"] = sec - t_end
    return out


def sharp_floor(fr, win, p):
    """Sharpness below which a frame counts as blurred: a fraction of the median at rest (else recovery, else all)."""
    for m in (win == "rest", win == "recovery", np.ones(len(fr), bool)):
        if np.isfinite(fr.sharpness[m].median()):
            return p["sharp_rel_min"] * fr.sharpness[m].median()


def plot_ecg(sid):
    if ECG[sid] is None:
        return
    E, seg = ECG[sid], segments[segments.session_id == sid]
    t_end = seg[seg.condition == "effort"].t_end_s.max()
    fig, ax = plt.subplots(3, 1, figsize=(10, 6.6))
    m = (E["t"] >= t_end - 10) & (E["t"] < t_end - 4)
    r = E["r"][m[E["r"]]]
    ax[0].plot(E["t"][m], E["y"][m], color=INK2, lw=0.8)
    ax[0].plot(E["t"][r], E["y"][r], "o", ms=4, color=SERIES[1])
    ax[0].set_title(f"{sid} — filtered ECG at the end of effort, detected R-peaks", loc="left")
    rr = E["rr"]
    ax[1].plot(rr.t[rr.ok], 60 / rr.rr[rr.ok], ".", ms=2, color=SERIES[0], label="accepted beats")
    ax[1].plot(rr.t[~rr.ok], (60 / rr.rr[~rr.ok]).clip(30, 230), "x", ms=4, color=SERIES[7], label="rejected")
    ax[1].set_title("Heart rate (bpm) against video time (s): must peak at the end of effort (dashed)", loc="left")
    ax[1].legend(loc="upper left", ncol=2)
    unit = lambda v: (v - np.nanpercentile(v, 5)) / (np.nanpercentile(v, 95) - np.nanpercentile(v, 5) + 1e-12)
    ax[2].plot(E["motion"].index, unit(E["motion"]), color=SERIES[1], lw=1, label="head motion (video)")
    ax[2].plot(E["noise"].index, unit(E["noise"]), color=SERIES[0], lw=1, label="motion noise of the ECG")
    ax[2].set(ylim=(-0.3, 1.6), xlabel="video time (s)")
    ax[2].set_title("Synchronisation check: both are driven by pedalling and must start and stop together", loc="left")
    ax[2].legend(loc="upper left", ncol=2)
    for a in ax[1:]:
        a.set_xlim(0, FRAMES[sid].t_s.max())
        for x in (seg[seg.condition == "effort"].t_start_s.min(), t_end):
            a.axvline(x, color=INK2, lw=0.8, ls="--")
    fig.tight_layout()
    show(fig, f"ecg_qc_{sid}.png")


def plot_rois(sid):
    fr = FRAMES[sid]
    lab = label_seconds(np.arange(int(fr.t_s.max()) + 1), segments[segments.session_id == sid], PARAMS)
    cap = cv2.VideoCapture(os.path.join(DATA_DIR, sessions.loc[sid, "video_path"]))
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
    for ax, c in zip(axes, COND):
        ax.axis("off")
        secs = lab.index[lab.window == c]
        row = fr.iloc[(fr.t_s - secs[len(secs) // 2]).abs().argmin()] if len(secs) else None
        if row is None or not np.isfinite(row.box_w):
            continue
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame))
        img, box = turn(cap.read()[1], ROTATE[sid]).copy(), (row.box_x, row.box_y, row.box_w, row.box_h)
        if ANONYMIZE_FIGURES:                                      # pixelate the face before drawing the ROIs
            face = roi(img, box, (-0.05, -0.2, 1.05, 1.2))
            face[:] = cv2.resize(cv2.resize(face, (12, 14)), face.shape[1::-1], interpolation=cv2.INTER_NEAREST)
        x, y, w, h = box
        for fx0, fy0, fx1, fy1 in [(0, 0, 1, 1)] + list(PARAMS["roi"].values()):
            cv2.rectangle(img, (int(x + fx0 * w), int(y + fy0 * h)), (int(x + fx1 * w), int(y + fy1 * h)), (0, 200, 255), 2)
        ax.imshow(cv2.cvtColor(roi(img, box, (-0.3, -0.3, 1.3, 1.3)), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{sid} — {c} (t = {row.t_s:.0f} s)")
    cap.release()
    show(fig, f"roi_check_{sid}.png")


qc = []
for sid, fr in FRAMES.items():
    plot_ecg(sid)
    plot_rois(sid)
    win = label_seconds(np.floor(fr.t_s).astype(int).values, segments[segments.session_id == sid], PARAMS).window.values
    sharp_min = sharp_floor(fr, win, PARAMS)
    for c in COND:
        g = fr[win == c]
        qc.append(dict(session=sid, condition=c, frames=len(g), detection_rate=g.det.mean(),
                       box_available=g.box_w.notna().mean(), sharp_ok=(g.sharpness >= sharp_min).mean(),
                       forehead_skin=g.skin_frac.mean()))
QC = pd.DataFrame(qc)
QC.to_csv(os.path.join(TAB_DIR, "quality_control.csv"), index=False)
display(QC.round(3))

## 6. Per-second features → `features.csv`
Per-frame features are aggregated per second (mean, SD, min, max in `features_second_stats.csv`; the mean in
`features.csv`, **one row per second, one column per feature**, with labels). Added per second:
`head_jitter` = SD of the raw box centre (% face width; its resting value is the detector's noise floor) and the
ECG heart rate `hr_ecg`.

In [ ]:

def aggregate(sid):
    """Per-second features (mean; mean, SD, min, max in a second table), head jitter, heart rate and labels."""
    fr, r, seg = FRAMES[sid].copy(), sessions.loc[sid], segments[segments.session_id == sid]
    fr["sec"] = np.floor(fr.t_s).astype(int)
    g = fr.groupby("sec")
    stats4 = g[[f for f in FEATURES if f in fr] + ["sharpness", "skin_frac"]].agg(["mean", "std", "min", "max"])
    out = stats4.xs("mean", axis=1, level=1).copy()
    out["head_jitter"] = 100 * np.hypot(g.raw_cx.std(), g.raw_cy.std()) / g.box_w.median()
    out["det_rate"], out["n_frames"] = g.det.mean(), g.size()
    out = out.join(ECG[sid]["hr"] if ECG[sid] else pd.Series(name="hr_ecg", dtype=float))
    out = pd.concat([label_seconds(out.index.values, seg, PARAMS), out], axis=1)
    out.insert(0, "t_s", out.index)
    stats4.columns = ["__".join(c) for c in stats4.columns]
    return out.assign(session_id=sid, subject_id=r.subject_id), stats4.reset_index().assign(session_id=sid)


parts = [aggregate(sid) for sid in sessions.index]
SEC = pd.concat([a for a, _ in parts], ignore_index=True)
SEC["hr_pct_max"] = 100 * SEC.hr_ecg / SEC.groupby("subject_id").hr_ecg.transform(
    lambda h: h.rolling(5, center=True, min_periods=3).mean().max())      # measured maximum (maximal tests)
COLS = (["session_id", "subject_id", "t_s", "segment", "condition", "stage", "power_W", "borg_cr10", "window",
         "stage_window", "t_since_end", "hr_ecg", "hr_pct_max"] + FEATURES + ["sharpness", "skin_frac",
                                                                              "det_rate", "n_frames"])
SEC[COLS].to_csv(os.path.join(OUT_DIR, "features.csv"), index=False, float_format="%.5g")
pd.concat([b for _, b in parts]).to_csv(os.path.join(OUT_DIR, "features_second_stats.csv"), index=False, float_format="%.5g")
print(SEC.shape, "-> features.csv")

## 7. Descriptive statistics and time course
Mean, SD, minimum and maximum of each feature over the per-second values of each analysis window; time course of
every feature (thin line: 1-s values; thick line: 10-s moving mean); box plots per condition.

In [ ]:
DESC = (SEC.dropna(subset=["window"]).melt(["session_id", "window"], ALL, "feature")
        .groupby(["session_id", "feature", "window"]).value.agg(["count", "mean", "std", "min", "max"]).reset_index())
DESC.to_csv(os.path.join(TAB_DIR, "descriptives.csv"), index=False)
display(DESC.pivot_table(index="feature", columns=["session_id", "window"], values="mean")
        .reindex(columns=list(COND), level=1).reindex(ALL).round(2))


def plot_timeline(sid):
    d, seg = SEC[SEC.session_id == sid].set_index("t_s"), segments[segments.session_id == sid]
    feats = ["hr_ecg"] + [f for f in FEATURES if d[f].notna().any()]
    fig, axes = plt.subplots(len(feats) + 2, 1, figsize=(10, 1.25 * (len(feats) + 2)), sharex=True)
    eff = seg[seg.condition == "effort"]
    for ax in axes:
        for s in eff[eff.stage % 2 == 1].itertuples():
            ax.axvspan(s.t_start_s, s.t_end_s, color="#efeee9", lw=0)
        for x in (eff.t_start_s.min(), eff.t_end_s.max()):
            ax.axvline(x, color=INK2, lw=0.8, ls="--")
    axes[0].step(d.index, d.power_W.fillna(0), where="post", color=INK)
    axes[0].set_title(f"{sid} — Power (W)   [dashed: start/end of exercise; shaded: odd stages]", loc="left")
    axes[1].plot(seg.t_end_s, seg.borg_cr10, "o", ms=5, color=INK)
    axes[1].set(ylim=(-0.5, 10.5))
    axes[1].set_title("Borg CR10 (end of each segment)", loc="left")
    for ax, f in zip(axes[2:], feats):
        ax.plot(d.index, d[f], color=SERIES[0], lw=0.5, alpha=0.45)
        ax.plot(d.index, d[f].rolling(10, center=True, min_periods=5).mean(), color=SERIES[0], lw=2)
        ax.set_title(f"{label(f)} ({FEATURE_INFO[f][1]})", loc="left")
    axes[-1].set_xlabel("video time (s)")
    fig.tight_layout()
    show(fig, f"timeline_{sid}.png")


def plot_boxes(sid):
    d = SEC[(SEC.session_id == sid) & SEC.window.notna()]
    feats = [f for f in ALL if d[f].notna().any()]
    nrow = int(np.ceil(len(feats) / 5))
    fig, axes = plt.subplots(nrow, 5, figsize=(13, 2.5 * nrow), squeeze=False)
    for ax in axes.flat[len(feats):]:
        ax.axis("off")
    for ax, f in zip(axes.flat, feats):
        bp = ax.boxplot([d.loc[d.window == c, f].dropna().values for c in COND], widths=0.55, patch_artist=True,
                        showfliers=False, medianprops=dict(color=INK, lw=1.5))
        for patch, c in zip(bp["boxes"], COND.values()):
            patch.set(facecolor=c, alpha=0.4, edgecolor=c, lw=1.5)
        ax.set_xticks([1, 2, 3], list(COND))
        ax.set_title(label(f))
        ax.set_ylabel(FEATURE_INFO[f][1], fontsize=7.5)
    fig.suptitle(f"{sid} — per-second values in each analysis window (★ = key feature)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    show(fig, f"boxplots_{sid}.png")


for sid in sessions.index:
    plot_timeline(sid)
    plot_boxes(sid)

## 8. Rest vs effort vs recovery
Each video contains one subject: inference is **within one test**. Consecutive seconds are strongly autocorrelated
(treating them as independent would be pseudo-replication) and pairing second *i* of rest with second *i* of effort
is arbitrary. Per-second values are therefore averaged into **non-overlapping 10-s blocks** (≥ 7 valid seconds),
the lag-1 autocorrelation of block residuals is checked, and blocks are compared with rank tests —
**Kruskal–Wallis** across the three conditions, **Mann–Whitney U** for each pair (Holm) — with **effect sizes and
95 % bootstrap CIs** (blocks resampled, fixed seed): Cliff's δ (−1 to +1), Hedges' g and the mean difference in the
feature's unit.

In [ ]:
def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    out = np.empty(len(p))
    out[order] = np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order]).clip(max=1)
    return out


def blocks(d, f, p):
    """Means of non-overlapping 10-s blocks of feature f in one window."""
    g = d.groupby((d.t_s - d.t_s.min()) // p["block_s"])[f]
    return g.mean()[g.count() >= p["min_valid_s"]].values


def cliffs_delta(b, a):
    return np.sign(b[:, None] - a[None, :]).mean()


def hedges_g(b, a):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (b.mean() - a.mean()) / sp * (1 - 3 / (4 * (na + nb) - 9)) if sp > 0 else np.nan


def compare_conditions(SEC, feats, p):
    rows, rng = [], np.random.default_rng(SEED)
    for sid, d in SEC.groupby("session_id"):
        for f in feats:
            B = {c: b for c, b in ((c, blocks(d[d.window == c], f, p)) for c in COND) if len(b) >= 2}
            if len(B) < 2:                                         # e.g. a video without rest: the other pair is kept
                continue
            res = np.concatenate([b - b.mean() for b in B.values()])
            try:
                kw_p = stats.kruskal(*B.values()).pvalue
            except ValueError:                                     # all values identical
                kw_p = np.nan
            for c1, c0 in (("effort", "rest"), ("recovery", "rest"), ("recovery", "effort")):
                if c1 not in B or c0 not in B:
                    continue
                b, a = B[c1], B[c0]
                boot = np.array([(bb.mean() - aa.mean(), cliffs_delta(bb, aa)) for bb, aa in
                                 ((rng.choice(b, len(b)), rng.choice(a, len(a))) for _ in range(p["n_boot"]))])
                (d_lo, c_lo), (d_hi, c_hi) = np.percentile(boot, [2.5, 97.5], axis=0)
                rows.append(dict(session=sid, feature=f, family=FEATURE_INFO[f][2], contrast=f"{c1} vs {c0}",
                                 n_blocks=f"{len(b)}/{len(a)}", mean_1=b.mean(), mean_0=a.mean(), diff=b.mean() - a.mean(),
                                 diff_lo=d_lo, diff_hi=d_hi, cliff_delta=cliffs_delta(b, a), delta_lo=c_lo, delta_hi=c_hi,
                                 hedges_g=hedges_g(b, a), kw_p=kw_p, mwu_p=stats.mannwhitneyu(b, a).pvalue,
                                 lag1_acf=np.corrcoef(res[1:], res[:-1])[0, 1]))
    out = pd.DataFrame(rows)
    out["mwu_p_holm"] = out.groupby(["session", "feature"]).mwu_p.transform(holm)        # 3 pairs per feature
    kw = out.drop_duplicates(["session", "feature"]).copy()
    kw["kw_p_holm"] = kw.groupby(["session", "family"]).kw_p.transform(lambda q: holm(q.fillna(1)))   # within family
    return out.merge(kw[["session", "feature", "kw_p_holm"]])


COMP = compare_conditions(SEC, ALL, PARAMS)
COMP.to_csv(os.path.join(TAB_DIR, "comparisons.csv"), index=False)
display(COMP[COMP.contrast == "effort vs rest"].drop(columns=["contrast", "mean_0", "mean_1", "mwu_p"]).round(3))
print("Median lag-1 autocorrelation of block residuals:", round(COMP.lag1_acf.median(), 2),
      "(> 0.3: blocks are not independent, read the p-values with caution)")

fig, axes = plt.subplots(1, 2, figsize=(11, 0.33 * len(ALL) + 1.3), sharey=True)
for ax, ct in zip(axes, ("effort vs rest", "recovery vs rest")):
    for k, sid in enumerate(sessions.index):
        d = COMP[(COMP.contrast == ct) & (COMP.session == sid)].set_index("feature").reindex(ALL)
        err = np.nan_to_num(np.clip([d.cliff_delta - d.delta_lo, d.delta_hi - d.cliff_delta], 0, None))
        ax.errorbar(d.cliff_delta, np.arange(len(ALL)) + 0.22 * (k - 0.5), xerr=err, fmt="o", ms=4, color=SERIES[k],
                    elinewidth=1.2, label=sid)
    ax.axvline(0, color=INK2, lw=0.8)
    ax.set(xlim=(-1.05, 1.05), xlabel="Cliff's δ (−1: always lower … +1: always higher)")
    ax.set_title(f"{ct}, 95 % bootstrap CI", loc="left")
axes[0].set_yticks(range(len(ALL)), [label(f) for f in ALL])
axes[0].invert_yaxis()
axes[0].legend(title="test", loc="lower left")
fig.tight_layout()
show(fig, "effect_sizes.png")

## 9. Dose–response, primary outcome and intensity domains
Each stage contributes one point: the mean of its **last 30 s** (rest = stage 0; an incomplete last stage counts if
it lasted ≥ 15 s). Within each test: Spearman's ρ, permutation p-value (9 999 permutations, fixed seed) and Fisher-z
CI (Bonett–Wright SE). When several tests have the feature, the **repeated-measures correlation** (rmcorr; Bakdash
& Marusich 2017, used by Timme & Brand 2020) gives the common within-test association; the clusters are the tests.
Stages are assigned to the **moderate / heavy / severe** domains with the powers at VT1 and VT2 of each participant
(measured in the masked test).

In a ramp, intensity and time rise together: any feature that drifts will correlate with Borg. The intensity
domains (§9.4) and the comparison with the recovery (§8) are what separates exertion from mere elapsed time.

In [ ]:
def spearman(x, y, n_perm, seed=SEED):
    """Spearman rho, permutation p-value, 95 % CI (Fisher z, Bonett-Wright SE), n."""
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, n = stats.rankdata(x[ok]), stats.rankdata(y[ok]), int(ok.sum())
    if n < 5 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return dict(rho=np.nan, p=np.nan, lo=np.nan, hi=np.nan, n=n)

    def pearson(a, b, axis=-1):                                    # Spearman = Pearson on ranks
        a, b = a - a.mean(axis, keepdims=True), b - b.mean(axis, keepdims=True)
        return (a * b).sum(axis) / np.sqrt((a * a).sum(axis) * (b * b).sum(axis))

    rho = float(pearson(x, y))
    pv = stats.permutation_test((x, y), pearson, permutation_type="pairings", vectorized=True, n_resamples=n_perm,
                                random_state=seed).pvalue
    lo, hi = np.tanh(np.arctanh(np.clip(rho, -0.999, 0.999)) + np.array([-1.96, 1.96]) * np.sqrt((1 + rho ** 2 / 2) / (n - 3)))
    return dict(rho=rho, p=pv, lo=lo, hi=hi, n=n)


def rmcorr(d, x, y, cluster="session_id"):
    """Repeated-measures correlation: Pearson r of the within-cluster centred values, dof = N - k - 1."""
    d = d[[x, y, cluster]].dropna()
    d = d[d.groupby(cluster)[x].transform("count") >= 3]
    k, dof = d[cluster].nunique(), len(d) - d[cluster].nunique() - 1
    if k < 2 or dof < 3:
        return dict(rho=np.nan, p=np.nan, lo=np.nan, hi=np.nan, n=len(d))
    r = np.corrcoef(*(d[v] - d.groupby(cluster)[v].transform("mean") for v in (x, y)))[0, 1]
    lo, hi = np.tanh(np.arctanh(np.clip(r, -0.999, 0.999)) + np.array([-1.96, 1.96]) / np.sqrt(dof - 1))
    return dict(rho=r, p=2 * stats.t.sf(abs(r) * np.sqrt(dof / max(1e-12, 1 - r ** 2)), dof), lo=lo, hi=hi, n=len(d))


STAGES = (SEC[SEC.stage_window & (SEC.stage >= 0)].groupby(["subject_id", "session_id", "stage"])
          [ALL + ["hr_pct_max", "power_W", "borg_cr10"]].mean().reset_index())
VT = sessions.reindex(columns=["vt1_W", "vt2_W"]).astype(float)
vt = VT.loc[STAGES.session_id].values
STAGES["domain"] = np.select([STAGES.stage == 0, STAGES.power_W < vt[:, 0], STAGES.power_W < vt[:, 1],
                              np.isfinite(vt[:, 1])], ["rest", "moderate", "heavy", "severe"], "n/a")
STAGES.to_csv(os.path.join(TAB_DIR, "stage_means.csv"), index=False)
REFS = ["borg_cr10", "hr_ecg", "power_W"]
DOSE = pd.DataFrame([dict(feature=f, reference=ref, level=lev, **res)
                     for f in ALL for ref in REFS if f != ref
                     for lev, res in [(sid, spearman(g[f].values, g[ref].values, PARAMS["n_perm"]))
                                      for sid, g in STAGES.groupby("session_id")] + [("rmcorr", rmcorr(STAGES, f, ref))]])
DOSE.to_csv(os.path.join(TAB_DIR, "dose_response.csv"), index=False)
display(DOSE.pivot_table(index="feature", columns=["reference", "level"], values="rho").reindex(ALL).round(2))

### 9.1 Primary outcome: mouth-opening area vs Borg CR10 and vs ECG heart rate

In [ ]:
PRIM = DOSE[(DOSE.feature == "mouth_area") & DOSE.reference.isin(["borg_cr10", "hr_ecg"])].dropna(subset=["rho"])
PRIM.to_csv(os.path.join(TAB_DIR, "primary_outcome.csv"), index=False)
display(PRIM.round(3))


def plot_dose(feats, ref, ref_label, name):
    """Stage means of each feature against a reference, one line per test, Spearman rho [95 % CI] in the legend."""
    fig, axes = plt.subplots(1, len(feats), figsize=(max(5, 3.3 * len(feats)), 3.3), squeeze=False)
    for ax, f in zip(axes[0], feats):
        for k, (sid, g) in enumerate(STAGES.groupby("session_id")):
            r = DOSE[(DOSE.feature == f) & (DOSE.reference == ref) & (DOSE.level == sid)].iloc[0]
            txt = f"{sid}: ρ = {r.rho:.2f} [{r.lo:.2f}, {r.hi:.2f}]" if np.isfinite(r.rho) else f"{sid}: not available"
            ax.plot(g[ref], g[f], "o-", ms=5, lw=1, color=SERIES[k], label=txt)
        ax.set_title(label(f))
        ax.set(xlabel=ref_label, ylabel=FEATURE_INFO[f][1])
        ax.legend(fontsize=6.5, loc="best")
    fig.tight_layout()
    show(fig, name)


plot_dose(["mouth_area"], "borg_cr10", "Borg CR10", "primary_outcome.png")
plot_dose(["mouth_area"], "hr_ecg", "ECG Heart Rate (bpm)", "primary_outcome_hr.png")

### 9.2 Borg CR10, heart rate and mouth opening against power (normalised)
To show on one axis variables with different units, the stage means of each variable are **min–max normalised
within each test**: 0 = lowest stage mean, 1 = highest stage mean of that test. Normalisation changes neither the
ranks nor the Spearman correlations, which are computed on the raw stage means (§9). One panel per test; dashed
lines: VT1 and VT2 of the participant. Statistics: Spearman ρ between mouth opening and Borg CR10 and between
mouth opening and heart rate, in each test and pooled (rmcorr).

In [ ]:
NORM_STYLE = {"borg_cr10": ("Borg CR10", SERIES[3]), "hr_ecg": ("ECG heart rate", SERIES[7]),
              "mouth_area": ("Mouth opening area", SERIES[0]), "forehead_a": ("Forehead redness a*", SERIES[4]),
              "head_jitter": ("Head instability", SERIES[2]), "power_W": ("power", INK)}


def minmax(v):
    """Min-max normalisation within one test: 0 = lowest stage mean, 1 = highest stage mean."""
    v = np.asarray(v, float)
    lo, hi = np.nanmin(v) if np.isfinite(v).any() else np.nan, np.nanmax(v) if np.isfinite(v).any() else np.nan
    return (v - lo) / (hi - lo) if hi > lo else np.full_like(v, np.nan)


def stat_table(pairs):
    """Spearman rho [95 % CI], p and n for each (feature, reference) pair, in each test and pooled (rmcorr)."""
    d = pd.concat([DOSE[(DOSE.feature == f) & (DOSE.reference == ref)] for f, ref in pairs])
    d["test"] = d.feature.map(lambda f: NORM_STYLE[f][0]) + " – " + d.reference.map(lambda r: NORM_STYLE[r][0])
    return d[["test", "level", "rho", "lo", "hi", "p", "n"]].reset_index(drop=True)


def plot_normalised(series, pairs, name, title):
    """Normalised stage means against power, one panel per test, VT1 / VT2 marked, Spearman results under each panel."""
    sids = list(STAGES.session_id.unique())
    tab = stat_table(pairs)
    fig, axes = plt.subplots(1, len(sids), figsize=(5.4 * len(sids), 4.6), sharey=True, squeeze=False)
    for ax, sid in zip(axes[0], sids):
        g = STAGES[STAGES.session_id == sid].sort_values("power_W")
        for f in series:
            lab, col = NORM_STYLE[f]
            y = minmax(g[f].values)
            if np.isfinite(y).any():
                ax.plot(g.power_W, y, "o-", ms=4.5, lw=1.4, color=col, label=lab)
            else:
                ax.plot([], [], "o-", color=col, label=f"{lab} (not available)")
        for v, nm in zip(VT.loc[sid], ("VT1", "VT2")):
            if np.isfinite(v):
                ax.axvline(v, color=INK2, ls="--", lw=0.8)
                ax.annotate(nm, (v, 1.0), xycoords=("data", "axes fraction"), ha="center", va="bottom", fontsize=7.5,
                            color=INK2)
        lines = [f"{r.test}: ρ = {r.rho:.2f} [{r.lo:.2f}, {r.hi:.2f}], p = {r.p:.3f}, n = {r.n}"
                 if np.isfinite(r.rho) else f"{r.test}: not available" for r in tab[tab.level == sid].itertuples()]
        ax.text(0.0, -0.21, "\n".join(lines), transform=ax.transAxes, va="top", fontsize=7)
        ax.set(xlabel="Power (W)", ylim=(-0.08, 1.08))
        ax.set_title(sid, loc="left", y=1.05)
        ax.legend(loc="upper left", fontsize=7)
    axes[0][0].set_ylabel("normalised stage mean (0 = min, 1 = max)")
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.33)
    show(fig, name)
    return tab


PAIRS_MOUTH = [("mouth_area", "borg_cr10"), ("mouth_area", "hr_ecg")]
STAT_MOUTH = plot_normalised(["borg_cr10", "hr_ecg", "mouth_area"], PAIRS_MOUTH, "normalised_borg_hr_mouth.png",
                             "Borg CR10, heart rate and mouth opening against power (min–max normalised within each test)")
STAT_MOUTH.to_csv(os.path.join(TAB_DIR, "spearman_mouth.csv"), index=False)
display(STAT_MOUTH.round(3))

### 9.3 Head instability and forehead redness against power (normalised)
Same normalisation and layout. Statistics (secondary outcomes): Spearman ρ between head instability and power,
between forehead redness and power, and between forehead redness and heart rate.

In [ ]:
PAIRS_SECONDARY = [("head_jitter", "power_W"), ("forehead_a", "power_W"), ("forehead_a", "hr_ecg")]
STAT_SECONDARY = plot_normalised(["head_jitter", "forehead_a"], PAIRS_SECONDARY, "normalised_head_redness.png",
                                 "Head instability and forehead redness against power (min–max normalised within each test)")
STAT_SECONDARY.to_csv(os.path.join(TAB_DIR, "spearman_secondary.csv"), index=False)
display(STAT_SECONDARY.round(3))

### 9.4 Intensity domains (VT1, VT2)
Mean of the stage means in each domain: **moderate** (power < VT1), **heavy** (VT1 ≤ power < VT2) and **severe**
(power ≥ VT2). H4 predicts a steeper rise of mouth opening above VT1 and VT2.

In [ ]:
if (STAGES.domain != "n/a").any():
    DOMAINS = STAGES.groupby(["session_id", "domain"])[ALL + ["borg_cr10"]].mean().reindex(
        ["rest", "moderate", "heavy", "severe"], level=1)
    DOMAINS.to_csv(os.path.join(TAB_DIR, "intensity_domains.csv"))
    display(DOMAINS.round(2).T)
else:
    print("No VT1 / VT2 given: intensity domains skipped.")

## 10. Discussion
*Scaffold: replace each bracket with what your own tables and figures show; never write a conclusion that a table
does not support.* The cell below gathers the key numbers.

**Primary outcome.** [ρ (95 % CI, p) between mouth-opening area and Borg CR10, and between mouth-opening area and
heart rate, in each test, and the pooled repeated-measures correlation — `primary_outcome.csv`,
`spearman_mouth.csv`, `normalised_borg_hr_mouth.png`.]
State plainly whether ρ > 0 is supported, and compare with r = 0.70 for *mouth open* vs RPE in Timme & Brand (2020).

**Secondary outcomes.** [Head instability vs power, forehead redness vs power and vs heart rate —
`spearman_secondary.csv`, `normalised_head_redness.png`.] [Features with |Cliff's δ| ≥ 0.47 and a CI excluding 0
between rest and effort — `comparisons.csv`.] [Behaviour across the moderate / heavy / severe domains: does mouth
opening rise more steeply above VT1 and VT2? — `intensity_domains.csv`.]

**What does not work, and why.** [Features without response.] Typical reasons: change smaller than the detector
noise (box jitter at rest), ROI drift with head motion, colour confounded by light or exposure, time constant longer
than a 1-min stage (redness).

**Correlation is not causation.** Exercise intensity is a common cause of heart rate, ventilation, mouth opening,
skin blood flow and head motion, and time is a second one (everything drifts during a ramp). The intensity domains
and the recovery window (§8) help to separate exertion from elapsed time. Other confounders: lighting and exposure,
head pose and distance, talking, the fan.

**Limitations.** Two participants, who are the authors: no generalisation, and no blinding to the hypotheses.
[Edit: order of the masked and unmasked tests and time between them — a test that follows a maximal test is not
performed fresh.] VT1/VT2 measured in the masked test are transferred to the unmasked test by power. One-minute
stages are not steady states. Viola–Jones gives a box, not landmarks: ROIs are fixed proportions of the box; beard,
glasses and hair change the masks. Borg CR10 is discrete and anchored by the previous answers.

**Improvements.** Facial landmarks (lip distance); tracking instead of per-frame detection; tests on separate days
and in random order; several subjects (multilevel models as in Timme & Brand 2020).

In [ ]:
print("PRIMARY OUTCOME — mouth area vs Borg CR10 and vs ECG heart rate:\n",
      STAT_MOUTH.round(3).to_string(index=False))
print("\nSECONDARY OUTCOMES — head instability and forehead redness:\n", STAT_SECONDARY.round(3).to_string(index=False))
key = COMP[(COMP.contrast == "effort vs rest") & COMP.feature.isin(KEY + ["hr_ecg"])]
print("\nEFFORT vs REST (Cliff's δ [95 % CI], Holm p):\n", key[["session", "feature", "cliff_delta", "delta_lo", "delta_hi",
                                                             "mwu_p_holm"]].round(3).to_string(index=False))

## 11. Reproducibility record
Same parameters for all sequences (`params.json`), fixed seed for every random step, per-frame features cached under
a hash of the parameters and of the video file, all figures and tables written by this notebook.

In [ ]:
print(json.dumps(dict(python=platform.python_version(), opencv=cv2.__version__, numpy=np.__version__, pandas=pd.__version__,
                      scipy=scipy.__version__, matplotlib=matplotlib.__version__, seed=SEED, params_hash=PARAM_HASH,
                      sessions=list(sessions.index), seconds=len(SEC)), indent=1))